In [ ]:
from lets_plot import LetsPlot, aes, geom_line, ggplot, ggsize, labs, scale_color_manual
import numpy as np
import pandas as pd

from ark.plot.theme import modern_theme
from ark.plot.tokens import BRAND_PALETTE

LetsPlot.setup_html()

In [ ]:
rng = np.random.default_rng(7)
minutes = np.arange(0, 24 * 60)
timestamp = pd.date_range("2026-01-15", periods=len(minutes), freq="1min")

# Baseline: lights and standby devices, roughly constant with small noise
baseline = 60 + rng.normal(0, 4, size=len(minutes))

# Fridge: cycles on/off roughly every 35 minutes, ~150W when running
cycle_len = 35
fridge_on = (minutes % cycle_len) < 18
fridge = np.where(fridge_on, 150, 0) + rng.normal(0, 3, size=len(minutes))

# Kettle: short high-power spikes at breakfast and dinner
kettle = np.zeros(len(minutes))
for start in (7 * 60 + 15, 18 * 60 + 40):
    kettle[start : start + 3] = 500

aggregate = baseline + fridge + kettle + rng.normal(0, 2, size=len(minutes))

signal = pd.DataFrame(
    {
        "timestamp": timestamp,
        "Aggregate": aggregate,
        "Baseline (lights, standby)": baseline,
        "Fridge": fridge,
        "Kettle": kettle,
    }
)
signal.head()

In [ ]:
long_df = signal.melt(id_vars="timestamp", var_name="Series", value_name="Power (W)")

plot = (
    ggplot(long_df, aes(x="timestamp", y="Power (W)", color="Series"))
    + geom_line(size=0.9)
    + scale_color_manual(values=BRAND_PALETTE)
    + labs(
        title="One meter, four unknowns",
        subtitle="The top line is all a smart meter ever reports. The other three are what NILM has to recover.",
        x="Time of day",
    )
    + modern_theme(legend_pos="bottom")
    + ggsize(700, 380)
)
plot